# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jvssankar/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My unit of analysis is one content-page record for the reporting period represented by that row. I will verify whether each row corresponds to a unique content page or whether pages can appear in multiple records. I will inspect the available date fields and the coverage of the reporting windows before claiming a specific date range. My ranking task is to prioritize pages for refresh review using information available for each record.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

data_url = (
    "https://raw.githubusercontent.com/"
    "Jvssankar/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(data_url)

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst five rows:")
display(df.head())

Dataset shape: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Data types:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc 

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I will separate the fields into features, label, context, and excluded fields. Candidate features include content type, content age, time since last update, and available search-performance measurements. The decline-label proxy is the label, not a feature. Content and client identifiers and reporting dates are context for grouping and auditing. Fields that directly define the label, contain future information, duplicate the target, or expose private information will be excluded from model inputs. I will confirm the exact field names and availability against the dataset schema before finalizing the contract.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Candidate fields, based on the research task
candidate_features = [
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "freshness_tier",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position"
]

label_fields = [
    "is_declining_label"
]

context_fields = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

# These must not be model inputs for this task
excluded_fields = {
    "content_id": "Identifier, not a predictive signal",
    "client_id": "Identifier; retain for group-based evaluation",
    "trend_direction": "Used to construct the target label",
    "trend_pct": "May directly encode the target trend",
    "is_declining_label": "Target, not an input feature"
}

# Keep only fields that actually exist
features = [c for c in candidate_features if c in df.columns]
labels = [c for c in label_fields if c in df.columns]
context = [c for c in context_fields if c in df.columns]

print("FEATURES:", features)
print("LABEL:", labels)
print("CONTEXT:", context)
print("\nEXCLUDED FIELDS AND REASONS:")

for field, reason in excluded_fields.items():
    if field in df.columns:
        print(f"- {field}: {reason}")

print("\nCandidate fields not found:")
print([
    c for c in candidate_features + label_fields + context_fields
    if c not in df.columns
])

FEATURES: ['content_type', 'content_age_days', 'days_since_last_update', 'freshness_tier', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position']
LABEL: []
CONTEXT: ['content_id', 'client_id', 'trend_direction', 'trend_pct']

EXCLUDED FIELDS AND REASONS:
- content_id: Identifier, not a predictive signal
- client_id: Identifier; retain for group-based evaluation
- trend_direction: Used to construct the target label
- trend_pct: May directly encode the target trend

Candidate fields not found:
['is_declining_label']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I will verify the data contract by checking the dataset dimensions, duplicate records, unique content and client counts, missing values, data types, and available date or reporting-window fields. I will also inspect the distribution of the target label. If the dataset contains multiple records for a content page, I will identify the appropriate reporting-period key before claiming that each page occurs only once. I will document any missing or unverifiable window information instead of assuming it.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Dataset size
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Exact duplicate rows:", df.duplicated().sum())

# 2. Unique identifiers
for col in ["content_id", "client_id"]:
    if col in df.columns:
        print(
            f"Unique {col}:",
            df[col].nunique(),
            "| Missing:",
            df[col].isna().sum()
        )

# 3. Check whether each content ID occurs only once
if "content_id" in df.columns:
    print(
        "Rows with repeated content_id:",
        df["content_id"].duplicated(keep=False).sum()
    )

# 4. Missing-value audit
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

display(
    missing.sort_values(
        "missing_percent",
        ascending=False
    ).head(20)
)

# 5. Inspect possible date/window columns
date_candidates = [
    c for c in df.columns
    if any(word in c.lower()
           for word in ["date", "time", "window", "period", "start", "end"])
]

print("Possible date/window fields:", date_candidates)

for col in date_candidates:
    print(f"\n{col}")
    print("Data type:", df[col].dtype)
    print("Distinct non-null values:", df[col].nunique())
    display(df[col].dropna().head(5))

# 6. Target-label distribution
if "is_declining_label" in df.columns:
    print("\nTarget counts:")
    print(df["is_declining_label"].value_counts(dropna=False))

    print("\nTarget percentages:")
    print(
        (df["is_declining_label"].value_counts(normalize=True) * 100)
        .round(2)
    )

Rows: 30000
Columns: 44
Exact duplicate rows: 0
Unique content_id: 30000 | Missing: 0
Unique client_id: 32 | Missing: 0
Rows with repeated content_id: 0


,missing_count,missing_percent
provider_used,21438,71.46
word_count,7699,25.66
char_count,7699,25.66
word_count_tier,7699,25.66
char_count_tier,7699,25.66
model_used,5733,19.11
trend_pct,3388,11.29
competition_level,2610,8.70
search_volume,2468,8.23
cpc,2468,8.23


Possible date/window fields: ['days_since_last_update', 'trend_direction', 'trend_pct']

days_since_last_update
Data type: int64
Distinct non-null values: 57


,days_since_last_update
0,20
1,25
2,20
3,22
4,14



trend_direction
Data type: object
Distinct non-null values: 5


,trend_direction
0,down
1,down
2,down
3,stable
4,down



trend_pct
Data type: float64
Distinct non-null values: 2712


,trend_pct
0,-41.4
1,-57.7
2,-60.9
3,-13.8
4,-34.7


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The dataset has limitations that affect interpretation. Pages may have unequal historical coverage, and some early records may contain GSC measurements without equivalent analytics data. Rolling reporting windows may overlap, so observations cannot automatically be treated as independent. Missing measurements may reflect unavailable coverage rather than poor performance. The decline label is rule-defined and does not establish the cause of a decline or prove future performance. I will use the available coverage and missingness checks to describe these limitations, and I will avoid causal claims or unsupported conclusions about dates that the data does not verify.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Identify fields relevant to coverage and limitations
keywords = [
    "source", "gsc", "search_console",
    "session", "history", "window",
    "date", "period", "start", "end",
    "impression", "click"
]

coverage_fields = [
    c for c in df.columns
    if any(word in c.lower() for word in keywords)
]

print("Fields relevant to coverage:")
print(coverage_fields)

# 2. Compare missingness across relevant fields
if coverage_fields:
    coverage_missing = pd.DataFrame({
        "missing_count": df[coverage_fields].isna().sum(),
        "missing_percent": (
            df[coverage_fields].isna().mean() * 100
        ).round(2)
    })

    display(
        coverage_missing.sort_values(
            "missing_percent",
            ascending=False
        )
    )

# 3. Check whether a source or history category exists
category_fields = [
    c for c in df.columns
    if any(word in c.lower()
           for word in ["source", "history", "coverage"])
]

for col in category_fields:
    print(f"\nDistribution for {col}:")
    display(df[col].value_counts(dropna=False).head(15))

# 4. Summarize available time-related fields
for col in [
    c for c in df.columns
    if any(word in c.lower()
           for word in ["date", "window", "period", "start", "end"])
]:
    print(f"\nTime-field check: {col}")
    print("Missing:", df[col].isna().sum())
    print("Distinct values:", df[col].nunique())
    print("Sample values:", df[col].dropna().head(5).tolist())

Fields relevant to coverage:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'days_since_last_update', 'impression_tier', 'trend_direction', 'trend_pct']


,missing_count,missing_percent
trend_pct,3388,11.29
impressions_90d,0,0.00
clicks_90d,0,0.00
engaged_sessions_90d,0,0.00
sessions_90d,0,0.00
days_with_impressions,0,0.00
days_with_sessions,0,0.00
impressions_last_30d,0,0.00
ai_sessions_90d,0,0.00
clicks_last_30d,0,0.00



Time-field check: days_since_last_update
Missing: 0
Distinct values: 57
Sample values: [20, 25, 20, 22, 14]

Time-field check: trend_direction
Missing: 0
Distinct values: 5
Sample values: ['down', 'down', 'down', 'stable', 'down']

Time-field check: trend_pct
Missing: 3388
Distinct values: 2712
Sample values: [-41.4, -57.7, -60.9, -13.8, -34.7]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.